# Los grupos BERT en el grafo de Ingrid

Los grupos son de PrimeKG y la estrella es de Ingrid. PrimeKG agrupa enfermedades de nombre parecido
con un modelo de lenguaje clínico (ClinicalBERT) y las fusiona: el grupo reemplaza a sus miembros,
que no existen como nodos aparte, y lleva todas sus aristas (en PrimeKG nativo los 1.267 grupos
tienen genes, fenotipos y drogas). Ingrid decidió no fusionar, para no alterar las enfermedades de
DisGeNET, que trata como fuente curada: conservó cada enfermedad como un CUI de UMLS y agregó el
grupo como un nodo conector, unido a sus miembros por una arista en estrella armada con las tablas de
referencia de PrimeKG (sección 3.3 y apéndice B de su tesis, `src/data/bert_edges_reference.py` de su
repositorio). En su grafo quedaron 1.040 grupos.

Este notebook mide, sobre el grafo de Ingrid, tres cosas que hacen falta para decidir si los grupos
entran o no en el experimento de proyecciones:

1. Qué aristas lleva el grupo y qué aristas llevan sus miembros.
2. Si el grupo y sus miembros comparten genes.
3. Cuánto cambia el plano de enfermedades y la capa gen-enfermedad si se los excluye.

Los datos se leen del contenedor `neo4j-thesis`, que tiene el grafo procesado de Ingrid
(`datos_ingriz/datos_tesis_ingrid/processed/`) cargado y verificado conteo por conteo contra sus
CSV. Ahí la estrella está separada como `DISEASE_BERT`, las aristas de MONDO como `DISEASE_DISEASE`,
las asociaciones gen-enfermedad de DisGeNET como `GDA`, y los grupos marcados con `es_grupo_bert`.
Nada de este notebook lee PrimeKG.

Requiere `docker start neo4j-thesis`.

In [1]:
import os
import sys

import networkx as nx
import pandas as pd
from neo4j import GraphDatabase

sys.path.insert(0, os.path.join(os.getcwd(), "neo4j"))
import config

pd.set_option("display.width", 160)
driver = GraphDatabase.driver(config.NEO4J_URI, auth=(config.NEO4J_USER, config.NEO4J_PASSWORD))


def consulta(cypher):
    '''Resultado de una consulta Cypher como DataFrame.'''
    registros, _, claves = driver.execute_query(cypher)
    return pd.DataFrame([r.values() for r in registros], columns=claves)

## 1. Los grupos y sus miembros

Cada arista `DISEASE_BERT` une un grupo con uno de sus miembros. Se verifica que todas vayan de un
grupo a un nodo que no es grupo, y cuántos miembros tiene cada grupo.

In [2]:
estrella = consulta(
    "MATCH (g:Disease)-[:DISEASE_BERT]-(m:Disease) WHERE g.es_grupo_bert "
    "RETURN g.node_index AS grupo, m.node_index AS miembro, m.es_grupo_bert AS miembro_es_grupo")

grupos_totales = consulta("MATCH (g:Disease {es_grupo_bert: true}) RETURN count(g) AS c").c[0]
aristas_estrella = consulta("MATCH ()-[r:DISEASE_BERT]->() RETURN count(r) AS c").c[0]
por_grupo = estrella.groupby("grupo").size()

print("grupos BERT:", grupos_totales, "; con al menos un miembro:", estrella.grupo.nunique())
print("aristas DISEASE_BERT:", aristas_estrella, "; filas grupo-miembro:", len(estrella))
print("aristas de la estrella que unen dos grupos:", int(estrella.miembro_es_grupo.sum()))
print("miembros distintos:", estrella.miembro.nunique(),
      "; miembros en mas de un grupo:", int((estrella.groupby("miembro").size() > 1).sum()))
print("miembros por grupo: mediana", por_grupo.median(), "; maximo", por_grupo.max())

grupos BERT: 1040 ; con al menos un miembro: 1040
aristas DISEASE_BERT: 4977 ; filas grupo-miembro: 4977
aristas de la estrella que unen dos grupos: 0
miembros distintos: 4977 ; miembros en mas de un grupo: 0
miembros por grupo: mediana 3.0 ; maximo 81


## 2. Qué aristas lleva cada uno

Para cada nodo `Disease`, cuántos genes tiene (`GDA`), cuántas aristas de MONDO (`DISEASE_DISEASE`) y
cuántas de la estrella (`DISEASE_BERT`). Se separan tres clases de nodo: los grupos, los CUIs que son
miembros de algún grupo, y los CUIs que no están en ningún grupo, que sirven de comparación.

In [3]:
grados = consulta(
    "MATCH (d:Disease) RETURN d.node_index AS nodo, d.es_grupo_bert AS es_grupo, "
    "COUNT { (d)-[:GDA]-() } AS genes, "
    "COUNT { (d)-[:DISEASE_DISEASE]-() } AS mondo, "
    "COUNT { (d)-[:DISEASE_BERT]-() } AS estrella")

miembros = set(estrella.miembro)
grados["clase"] = "CUI fuera de grupos"
grados.loc[grados.nodo.isin(miembros), "clase"] = "CUI miembro de un grupo"
grados.loc[grados.es_grupo, "clase"] = "grupo BERT"


def resumir(filas):
    return pd.Series({
        "nodos": len(filas),
        "con genes %": round(100 * (filas.genes > 0).mean(), 1),
        "genes mediana": filas.genes.median(),
        "con aristas MONDO %": round(100 * (filas.mondo > 0).mean(), 1),
        "aristas MONDO mediana": filas.mondo.median(),
        "aristas estrella mediana": filas.estrella.median(),
    })


tabla_clases = grados.groupby("clase")[["genes", "mondo", "estrella"]].apply(resumir).loc[
    ["grupo BERT", "CUI miembro de un grupo", "CUI fuera de grupos"]]
tabla_clases

,nodos,con genes %,genes mediana,con aristas MONDO %,aristas MONDO mediana,aristas estrella mediana
clase,,,,,,
grupo BERT,1040.0,0.0,0.0,89.9,2.0,3.0
CUI miembro de un grupo,4977.0,81.5,1.0,0.0,0.0,1.0
CUI fuera de grupos,10062.0,70.7,1.0,63.1,1.0,0.0


## 3. ¿El grupo y sus miembros comparten genes?

Si el grupo fuera un resumen de sus miembros, se esperaría que tuviera sus genes, o que los miembros
compartieran los genes del grupo. Para cada par grupo y miembro se compara el conjunto de genes de
uno y otro.

In [4]:
gda = consulta("MATCH (d:Disease)-[:GDA]-(g:Gene) RETURN d.node_index AS enfermedad, g.node_index AS gen")
genes_de = gda.groupby("enfermedad").gen.apply(set).to_dict()


def comparar(grupo, miembro):
    del_grupo = genes_de.get(grupo, set())
    del_miembro = genes_de.get(miembro, set())
    if not del_grupo and not del_miembro:
        return "los dos sin genes"
    if del_grupo == del_miembro:
        return "mismos genes"
    if not (del_grupo & del_miembro):
        return "genes disjuntos"
    return "comparten algunos genes"


comparacion = estrella.apply(lambda fila: comparar(fila.grupo, fila.miembro), axis=1)
comparacion.value_counts().rename("pares grupo-miembro").to_frame()

,pares grupo-miembro
genes disjuntos,4057
los dos sin genes,920


### Un ejemplo: la hipercolesterolemia

Todos los nodos cuyo nombre contiene "hypercholesterol", con su clase, sus genes, sus aristas de
MONDO y a qué grupo pertenecen o qué miembros tienen.

In [5]:
ejemplo = consulta(
    "MATCH (d:Disease) WHERE toLower(d.name) CONTAINS 'hypercholesterol' "
    "OPTIONAL MATCH (d)-[:DISEASE_BERT]-(otro:Disease) "
    "RETURN d.node_index AS nodo, d.id AS id, d.name AS nombre, d.es_grupo_bert AS es_grupo, "
    "COUNT { (d)-[:GDA]-() } AS genes, COUNT { (d)-[:DISEASE_DISEASE]-() } AS aristas_mondo, "
    "collect(otro.name) AS unido_por_la_estrella_a "
    "ORDER BY es_grupo DESC, genes DESC")
ejemplo

,nodo,id,nombre,es_grupo,genes,aristas_mondo,unido_por_la_estrella_a
0,1532,11369_7751,"hypercholesterolemia, autosomal dominant",True,0,1,"[Hyperlipoproteinemia Type IIb, HYPERCHOLESTER..."
1,1534,11374_7750_5439,familial hypercholesterolemia,True,0,4,"[Hypercholesterolemia, Familial, HYPERCHOLESTE..."
2,19667,C0020443,Hypercholesterolemia,False,39,0,[]
3,19668,C0020445,"Hypercholesterolemia, Familial",False,18,0,[familial hypercholesterolemia]
4,23905,C0342881,Familial hypercholesterolemia - homozygous,False,3,1,[]
5,28903,C1863551,"HYPERCHOLESTEROLEMIA, AUTOSOMAL DOMINANT, 3",False,1,0,"[hypercholesterolemia, autosomal dominant]"
6,32154,C3888316,"Hypercholesterolemia, familial, due to ligand-...",False,1,0,"[hypercholesterolemia, autosomal dominant]"
7,34172,C4751204,Hypercholesterolemia due to cholesterol 7alpha...,False,1,1,[]
8,28900,C1863512,"HYPERCHOLESTEROLEMIA, AUTOSOMAL RECESSIVE",False,1,0,[familial hypercholesterolemia]


## 4. Qué cuesta excluirlos

El plano de enfermedades del experimento es la red `Disease`-`Disease` donde corre DIAMOnD. Se
compara en tres versiones:

- el grafo de Ingrid tal cual (MONDO más la estrella, con los grupos),
- solo MONDO, con los grupos pero sin la estrella,
- solo MONDO y sin los grupos, que es lo que usa hoy el experimento.

Además, cuántas asociaciones gen-enfermedad caen sobre un miembro de algún grupo: en la celda que
va de un gen al plano de enfermedades, esas enfermedades solo pueden ser semillas si están en el plano.

In [6]:
aristas_plano = consulta(
    "MATCH (a:Disease)-[r:DISEASE_DISEASE|DISEASE_BERT]->(b:Disease) "
    "RETURN a.node_index AS a, b.node_index AS b, type(r) AS relacion, "
    "a.es_grupo_bert AS a_grupo, b.es_grupo_bert AS b_grupo")

versiones = {
    "tal cual (MONDO y estrella)": aristas_plano,
    "solo MONDO, con grupos": aristas_plano[aristas_plano.relacion == "DISEASE_DISEASE"],
    "solo MONDO, sin grupos (experimento)": aristas_plano[
        (aristas_plano.relacion == "DISEASE_DISEASE") & ~aristas_plano.a_grupo & ~aristas_plano.b_grupo],
}

filas = []
for nombre, aristas in versiones.items():
    red = nx.Graph()
    red.add_edges_from(zip(aristas.a, aristas.b))
    en_el_plano = set(red.nodes())
    filas.append({
        "plano disease": nombre,
        "nodos": red.number_of_nodes(),
        "aristas": red.number_of_edges(),
        "componente mayor": len(max(nx.connected_components(red), key=len)),
        "miembros de grupos en el plano": len(miembros & en_el_plano),
    })
pd.DataFrame(filas).set_index("plano disease")

,nodos,aristas,componente mayor,miembros de grupos en el plano
plano disease,,,,
tal cual (MONDO y estrella),12368,16433,11074,4977
"solo MONDO, con grupos",7286,11456,6844,0
"solo MONDO, sin grupos (experimento)",5318,7836,4644,0


In [7]:
en_miembros = gda.enfermedad.isin(miembros)
print("aristas GDA:", len(gda))
print("con la enfermedad miembro de un grupo:", int(en_miembros.sum()),
      f"({100 * en_miembros.mean():.1f} %)")
print("enfermedades con al menos un gen:", gda.enfermedad.nunique(),
      "; de ellas, miembros de un grupo:", gda[en_miembros].enfermedad.nunique())
driver.close()

aristas GDA: 84014
con la enfermedad miembro de un grupo: 17172 (20.4 %)
enfermedades con al menos un gen: 11166 ; de ellas, miembros de un grupo: 4057


## Lectura

**El grupo y sus miembros no comparten aristas, se las reparten.** Ningún grupo tiene genes y el
89,9 % tiene aristas de MONDO; ningún miembro tiene aristas de MONDO y el 81,5 % tiene genes. En los
4.977 pares grupo-miembro los genes nunca coinciden: 4.057 son conjuntos disjuntos (el miembro tiene
genes y el grupo no) y en 920 los dos están vacíos. El grupo funciona como la cara ontológica de la
enfermedad (su lugar en MONDO) y los miembros como la cara DisGeNET (sus genes), unidos solo por la
estrella. La estructura es limpia: cada miembro pertenece a un solo grupo y la estrella nunca une dos
grupos.

**El ejemplo lo muestra.** "familial hypercholesterolemia" es un grupo con 4 aristas de MONDO y
ningún gen; su miembro "Hypercholesterolemia, Familial" tiene 18 genes y ninguna arista de MONDO. En
cambio "Hypercholesterolemia" a secas, con 39 genes, no está en ningún grupo y tampoco tiene aristas de
MONDO: no todo CUI sin MONDO es miembro de un grupo. De los 10.062 CUIs fuera de grupos, el 36,9 %
no tiene aristas de MONDO, así que el plano de enfermedades pierde nodos también por fuera de los
grupos.

**Excluir los grupos saca a todos sus miembros del plano de enfermedades**, porque su única arista ahí
es la estrella. El plano pasa de 12.368 nodos y 16.433 aristas (tal cual) a 5.318 nodos y 7.836
aristas (el del experimento), y la componente mayor de 11.074 a 4.644 nodos. Quitar solo la estrella
dejando los grupos ya produce casi toda la pérdida de miembros: sin estrella ningún miembro queda en
el plano. El 20,4 % de las asociaciones gen-enfermedad (17.172 de 84.014) cae sobre un miembro; en
la celda que va de un gen al plano de enfermedades, esas enfermedades no pueden ser semillas.

Qué hacer con los grupos (excluirlos, dejarlos en el plano sin que sean anclas, o contraer cada grupo
con sus miembros) es una decisión para hablar con Ariel; este notebook solo mide.